# Semana 01 · AgentOps y LLMOps con MLflow

Práctica guiada para observar trazas y evaluación mínima sin depender de un proveedor LLM. Usaremos un agente determinista para aprender el flujo: entrada → ruta → contexto → respuesta → traza → evaluación.


## 0. Dependencias y configuración

Deja `USE_LLM = False`. El objetivo es instrumentar y evaluar, no llamar a un modelo externo.


In [0]:
%pip install -q --upgrade "mlflow[databricks]"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#depues de intsalr la celda anterior
%restart_python

In [0]:
import json
from typing import Any

import mlflow
#se crea el experimento
USE_LLM = False
AGENT_VERSION = "v1-deterministic"
EXPERIMENT_NAME = "/Shared/muiaap-operacion-modelos/semana01-agentops"
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.set_tags({"course": "operacion-modelos", "week": "01", "agent.version": AGENT_VERSION})

2026/10/02 21:23:42 INFO mlflow.tracking.fluent: Experiment with name '/Shared/muiaap-operacion-modelos/semana01-agentops' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog
2026/10/02 21:23:42 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


## 1. Trazar un agente sencillo

**TODO 1:** completa las reglas de ruta, recuperación y respuesta. Decide qué palabras activan cada ruta, porque esa decisión define el comportamiento observable. Inspecciona `@mlflow.trace`. Verifica que cada pregunta devuelve texto y que en MLflow aparecen spans anidados.


In [0]:
@mlflow.trace
def route_question(question: str) -> str: #que nos uba nada que no se deba subir
    text = question.lower()
    if any(word in text for word in ["token", "secreto", "contraseña", "privacidad"]):
        return "security"
    if any(word in text for word in ["producción", "api", "servicio"]):
        return "production"
    return "mlflow"


@mlflow.trace
def retrieve_course_context(topic: str) -> str: #dependiendo de la ruta devielve un taxto
    contexts = {
        "security": "No registres secretos en parámetros, tags ni artefactos.",
        "production": "En producción importan contratos, versionado y monitorización.",
        "mlflow": "Un experimento agrupa runs con parámetros, métricas y artefactos.",
    }
    return contexts[topic]


@mlflow.trace
def course_agent(question: str) -> str:
    route = route_question(question)
    context = retrieve_course_context(route)
    return f"Ruta: {route}. {context}"

questions = [
    "¿Qué objeto agrupa varios runs?",
    "¿Puedo pegar mi token en un parámetro de MLflow?",
    "¿Qué cambia cuando llevo un modelo a producción?",
]
answers = [{"question": q, "answer": course_agent(q)} for q in questions]
display(answers)
assert all("Ruta:" in item["answer"] for item in answers)

2026/10/02 21:30:10 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


answer,question
"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",¿Qué objeto agrupa varios runs?
"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",¿Puedo pegar mi token en un parámetro de MLflow?
"Ruta: production. En producción importan contratos, versionado y monitorización.",¿Qué cambia cuando llevo un modelo a producción?


[Trace(trace_id=tr-2745058ed77e6af5120b74e5815473e2), Trace(trace_id=tr-9b5e42c3e469686aebee112b218acef6), Trace(trace_id=tr-ba7ce3dbb5b5a0c0272ae106afaa7b1a)]

## 2. Inspeccionar una traza

En **Experiments**, abre una traza de `course_agent`. Anota qué ruta siguió la pregunta, qué contexto recuperó y qué cambiarías si una ruta fuese incorrecta.


## 3. Evaluación mínima y transparente

No usamos un juez LLM. Comprobamos una expectativa simple: que la respuesta contenga una palabra esperada.

**TODO 2:** añade al menos dos casos: uno de privacidad/secretos y otro de producción. Importa porque la evaluación debe cubrir comportamientos relevantes. Inspecciona listas de diccionarios y verifica que `evaluation_df` tiene todos los casos.


In [0]:
EVALUATION_DATA = [
    {"question": "¿Qué objeto agrupa varios runs?", "must_include": "experimento"},
    {"question": "¿Puedo guardar una contraseña como tag?", "must_include": "secretos"},
    {"question": "¿Qué debo cuidar en producción?", "must_include": "contratos"},
    #DONE TODO 2: añade dos casos más y comprueba que siguen pasando.
    {"question": "¿Es seguro registrar un token en MLflow?", "must_include": "secretos"},
    {"question": "¿Qué es importante al desplegar un servicio?", "must_include": "versionado"},
]

evaluation_rows = []
for case in EVALUATION_DATA:
    answer = course_agent(case["question"])
    passed = case["must_include"].lower() in answer.lower()
    evaluation_rows.append({**case, "answer": answer, "passed": passed})

evaluation_df = __import__("pandas").DataFrame(evaluation_rows)
display(evaluation_df)
assert evaluation_df["passed"].all(), "Alguna expectativa mínima no se cumple"

question,must_include,answer,passed
¿Qué objeto agrupa varios runs?,experimento,"Ruta: mlflow. Un experimento agrupa runs con parámetros, métricas y artefactos.",true
¿Puedo guardar una contraseña como tag?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué debo cuidar en producción?,contratos,"Ruta: production. En producción importan contratos, versionado y monitorización.",true
¿Es seguro registrar un token en MLflow?,secretos,"Ruta: security. No registres secretos en parámetros, tags ni artefactos.",true
¿Qué es importante al desplegar un servicio?,versionado,"Ruta: production. En producción importan contratos, versionado y monitorización.",true


[Trace(trace_id=tr-ab3816def2787af28ba9fb0fda4720fc), Trace(trace_id=tr-ec8bc35cebaf7f12c689c90690f3cd42), Trace(trace_id=tr-d8013694b2a79b7867100bb34a77be9d), Trace(trace_id=tr-2fd079fdb54eb87b019b799bab4b6ea6), Trace(trace_id=tr-fd0916fea4e4c0d8ec6cf1b9819d86d2)]

## 4. Registrar un run de evaluación

**TODO 3:** registra el número de casos y el porcentaje que pasa. Importa porque una evaluación sin run se pierde. Inspecciona `mlflow.log_metric` y `mlflow.log_dict`. Verifica que el run contiene el artefacto `evaluation/results.json`.


In [0]:
#antes d ejecutar la siguiente celda he tenido que usar esto por que hay un experimento nuning. Asi veo si lo hay y cual es su id
print(mlflow.active_run()) 

active_run = mlflow.active_run()

print("Run activo:", active_run)
print("Run ID:", active_run.info.run_id if active_run else None)


<ActiveRun: >
Run activo: <ActiveRun: >
Run ID: cfffcd3a4a8343a595f925f7bdbc0992


In [0]:
#cone esto paro el run del experimento

mlflow.end_run()
print(mlflow.active_run())

None


In [0]:
with mlflow.start_run(run_name=f"agent-evaluation-{AGENT_VERSION}") as run:
    mlflow.set_tags({"agent.version": AGENT_VERSION, "purpose": "guided-agentops-evaluation"})
    mlflow.log_param("case_count", len(evaluation_df)) #registra num casos evaluados
    mlflow.log_metric("pass_rate", float(evaluation_df["passed"].mean())) #registra % de los que superan
    mlflow.log_dict(evaluation_rows, "evaluation/results.json")#guarda resultados
    EVALUATION_RUN_ID = run.info.run_id

print({"evaluation_run_id": EVALUATION_RUN_ID, "pass_rate": float(evaluation_df["passed"].mean())})

2026/10/02 21:48:42 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


{'evaluation_run_id': 'cf502bd554ad44c48b547f4bfa79af01', 'pass_rate': 1.0}


## 5. Buscar trazas o runs por versión

**TODO 4:** filtra por `agent.version`. Importa porque un agente cambia rápido y necesitas comparar versiones. Inspecciona `mlflow.search_runs` y, si tu workspace lo permite, `mlflow.search_traces`. Verifica que sólo recuperas la versión `v1-deterministic`.


In [0]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.agent.version = '{AGENT_VERSION}'",
    order_by=["attributes.start_time DESC"],
)
display(runs[["run_id", "tags.agent.version", "metrics.pass_rate"]])
assert set(runs["tags.agent.version"].dropna()) == {AGENT_VERSION}

run_id,tags.agent.version,metrics.pass_rate
cf502bd554ad44c48b547f4bfa79af01,v1-deterministic,1.0
cfffcd3a4a8343a595f925f7bdbc0992,v1-deterministic,null


## Entregable

Incluye en la ficha: una captura de una traza, `EVALUATION_RUN_ID`, el `pass_rate` y una frase sobre qué riesgo cubrirías en la siguiente versión del agente.

Evaluation_run_Id = cf502bd554ad44c48b547f4bfa79af01
PAss rate = 1

Se podria ver que hacer si una pregunta es clasificada en el grupo incorrecto.